# Merge modelling panel

Aggregate interim vector / virus tables into a **pathogen-long** supervised panel (specs `03` / `04`).

| Item | Detail |
|------|--------|
| Inputs | Interim vectors / viruses + `pathogen_vector_group.csv` + `priority_vector_taxa.csv` |
| Outputs | `data/03_processed/modelling_panel.parquet` + `modelling_panel_manifest.json` |
| Row semantics (B2) | Observed VectorNet `(spatial key, year, month, vector_group)` **×** each mapped `pathogen_name` |
| Zone clock (A3) | `first_vector_*` per cell×`vector_group` (vector establishment; not per pathogen) |
| Primary label | `pathogen_within_5y` (forward ≤5y, same key × **this** pathogen) |
| Secondary | `pathogen_within_10y`; ever / delay / same-month; source & geo shares for this pathogen |
| Species features | `species_richness` + `vector_count_<slug>` from priority taxa |
| Tier 1 | `(h3_cell, year, month, vector_group, pathogen_name)` |
| Tier 2 | `(country_iso2, year, month, vector_group, pathogen_name)` |
| Training filter | **VectorNet coverage required** — virus-only units excluded; no synthetic empty months |
| Ecology | Explode / label only within the same `vector_group` |

Country-only virus rows do **not** invent H3 and do not contribute to Tier‑1 labels.



In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from vector_guard.data import (
    default_processed_dir,
    default_reference_dir,
    default_vector_raw_dir,
    default_virus_raw_dir,
    find_repo_root,
)
from vector_guard.resolve import (
    GEO_PRECISION_CENTROID,
    GEO_PRECISION_EXACT,
    H3_RESOLUTION,
    load_interim_vectors,
    load_interim_viruses,
    load_pathogen_vector_group,
    load_priority_vector_taxa,
)
from vector_guard.virus_extract import INSDC_SOURCES, OUTBREAK_SOURCES

REPO_ROOT = find_repo_root(Path.cwd())
REF_DIR = default_reference_dir(REPO_ROOT)
PROCESSED_DIR = default_processed_dir(REPO_ROOT)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PANEL_PATH = PROCESSED_DIR / "modelling_panel.parquet"
MANIFEST_PATH = PROCESSED_DIR / "modelling_panel_manifest.json"

REPO_ROOT, PANEL_PATH, MANIFEST_PATH, H3_RESOLUTION


## Load interim sides + reference maps

In [ ]:
vectors = load_interim_vectors()
viruses = load_interim_viruses()
pathogen_map = load_pathogen_vector_group(REF_DIR)
priority_taxa = load_priority_vector_taxa(REF_DIR)

assert not pathogen_map["pathogen_name"].duplicated().any()
assert not priority_taxa["feature_slug"].duplicated().any()
assert set(pathogen_map["vector_group"]) <= {"mosquito", "tick", "biting_midge", "sandfly"}

{
    "vectors": len(vectors),
    "viruses": len(viruses),
    "pathogens": len(pathogen_map),
    "priority_taxa": len(priority_taxa),
    "H3_RESOLUTION": H3_RESOLUTION,
}


## Filter to panel-key-ready rows

In [ ]:
def _ready(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[
        (df["vector_group"].fillna("") != "")
        & df["year"].notna()
        & df["month"].notna()
    ].copy()


vectors_r = _ready(vectors)
viruses_r = _ready(viruses)

vectors_r["year"] = vectors_r["year"].astype(int)
vectors_r["month"] = vectors_r["month"].astype(int)
viruses_r["year"] = viruses_r["year"].astype(int)
viruses_r["month"] = viruses_r["month"].astype(int)

# Virus events need pathogen_name for pathogen-specific labels
viruses_r = viruses_r.loc[viruses_r["pathogen_name"].fillna("") != ""].copy()

vectors_t1 = vectors_r.loc[vectors_r["h3_cell"].notna()].copy()
vectors_t2 = vectors_r.loc[
    vectors_r["h3_cell"].isna() & (vectors_r["country_iso2"].fillna("") != "")
].copy()

viruses_t1 = viruses_r.loc[viruses_r["h3_cell"].notna()].copy()
viruses_t2 = viruses_r.loc[
    viruses_r["h3_cell"].isna() & (viruses_r["country_iso2"].fillna("") != "")
].copy()

{
    "vectors_ready": len(vectors_r),
    "vectors_tier1_rows": len(vectors_t1),
    "vectors_tier2_rows": len(vectors_t2),
    "viruses_ready": len(viruses_r),
    "viruses_tier1_rows": len(viruses_t1),
    "viruses_tier2_rows": len(viruses_t2),
}


## Aggregate helpers (species features + pathogen-long labels)

In [ ]:
HORIZON_YEARS_PRIMARY = 5
HORIZON_YEARS_SECONDARY = 10

PRIORITY_SLUGS = priority_taxa["feature_slug"].tolist()
PRIORITY_NAME_TO_SLUG = dict(
    zip(priority_taxa["scientific_name"], priority_taxa["feature_slug"], strict=True)
)


def _parse_individual_count(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def _ym_index(year: pd.Series, month: pd.Series) -> pd.Series:
    return year.astype(int) * 12 + (month.astype(int) - 1)


def _ym_parts(ym: pd.Series) -> tuple[pd.Series, pd.Series]:
    year = ym // 12
    month = (ym % 12) + 1
    return year.astype(int), month.astype(int)


def _mode_country(s: pd.Series) -> str:
    cleaned = s.fillna("").replace("", pd.NA).dropna()
    if cleaned.empty:
        return ""
    return str(cleaned.mode().iloc[0])


def _priority_count_frame(work: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    """Wide detection counts for priority taxa (0 if absent)."""
    tagged = work.copy()
    tagged["_slug"] = tagged["scientific_name"].map(PRIORITY_NAME_TO_SLUG)
    tagged = tagged.loc[tagged["_slug"].notna()]
    if tagged.empty:
        base = work.groupby(keys, dropna=False).size().rename("_n").reset_index()[keys]
        for slug in PRIORITY_SLUGS:
            base[f"vector_count_{slug}"] = 0
        return base
    wide = (
        tagged.groupby(keys + ["_slug"], dropna=False)
        .size()
        .unstack("_slug", fill_value=0)
        .reset_index()
    )
    for slug in PRIORITY_SLUGS:
        col = slug
        if col not in wide.columns:
            wide[col] = 0
        wide[f"vector_count_{slug}"] = wide[col].astype(int)
    drop_cols = [c for c in wide.columns if c in PRIORITY_SLUGS]
    return wide.drop(columns=drop_cols)


def aggregate_vectors_tier1(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["_ic"] = _parse_individual_count(work["individual_count"])
    work["_ym"] = _ym_index(work["year"], work["month"])
    keys = ["h3_cell", "year", "month", "vector_group"]
    out = (
        work.groupby(keys, dropna=False)
        .agg(
            vector_detection_count=("record_id", "size"),
            individual_count_sum=("_ic", "sum"),
            individual_count_mean=("_ic", "mean"),
            country_iso2=("country_iso2", _mode_country),
            species_richness=("scientific_name", "nunique"),
        )
        .reset_index()
    )
    priority = _priority_count_frame(work, keys)
    out = out.merge(priority, on=keys, how="left")
    for slug in PRIORITY_SLUGS:
        col = f"vector_count_{slug}"
        out[col] = out[col].fillna(0).astype(int)
    first = (
        work.groupby(["h3_cell", "vector_group"], dropna=False)["_ym"]
        .min()
        .rename("_first_ym")
        .reset_index()
    )
    fy, fm = _ym_parts(first["_first_ym"])
    first["first_vector_year"] = fy
    first["first_vector_month"] = fm
    out = out.merge(
        first[["h3_cell", "vector_group", "first_vector_year", "first_vector_month"]],
        on=["h3_cell", "vector_group"],
        how="left",
    )
    out["spatial_tier"] = "h3"
    out["vector_present"] = 1
    out["individual_count_sum"] = out["individual_count_sum"].fillna(0.0)
    return out


def aggregate_vectors_tier2(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["_ic"] = _parse_individual_count(work["individual_count"])
    work["_ym"] = _ym_index(work["year"], work["month"])
    keys = ["country_iso2", "year", "month", "vector_group"]
    out = (
        work.groupby(keys, dropna=False)
        .agg(
            vector_detection_count=("record_id", "size"),
            individual_count_sum=("_ic", "sum"),
            individual_count_mean=("_ic", "mean"),
            species_richness=("scientific_name", "nunique"),
        )
        .reset_index()
    )
    priority = _priority_count_frame(work, keys)
    out = out.merge(priority, on=keys, how="left")
    for slug in PRIORITY_SLUGS:
        col = f"vector_count_{slug}"
        out[col] = out[col].fillna(0).astype(int)
    first = (
        work.groupby(["country_iso2", "vector_group"], dropna=False)["_ym"]
        .min()
        .rename("_first_ym")
        .reset_index()
    )
    fy, fm = _ym_parts(first["_first_ym"])
    first["first_vector_year"] = fy
    first["first_vector_month"] = fm
    out = out.merge(
        first[["country_iso2", "vector_group", "first_vector_year", "first_vector_month"]],
        on=["country_iso2", "vector_group"],
        how="left",
    )
    out["spatial_tier"] = "country"
    out["h3_cell"] = pd.NA
    out["vector_present"] = 1
    out["individual_count_sum"] = out["individual_count_sum"].fillna(0.0)
    return out


def explode_pathogens(vec_panel: pd.DataFrame, pathogens: pd.DataFrame) -> pd.DataFrame:
    """Left-merge every pathogen mapped to the unit's vector_group."""
    return vec_panel.merge(pathogens, on="vector_group", how="inner")


def _prepare_virus_events(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["_v_ym"] = _ym_index(work["year"], work["month"])
    work["_insdc"] = work["source"].isin(INSDC_SOURCES).astype(int)
    work["_outbreak"] = work["source"].isin(OUTBREAK_SOURCES).astype(int)
    work["_exact"] = (work["geo_precision"] == GEO_PRECISION_EXACT).astype(float)
    work["_centroid"] = (work["geo_precision"] == GEO_PRECISION_CENTROID).astype(float)
    return work


def attach_b2_pathogen_labels(
    vec_panel: pd.DataFrame,
    viruses: pd.DataFrame,
    *,
    spatial_key: str,
    include_geo_shares: bool,
) -> pd.DataFrame:
    """Attach pathogen-specific B2 forward / ever / delay labels (specs 03/04)."""
    out = vec_panel.copy().reset_index(drop=True)
    out["_row_id"] = out.index
    out["_p_ym"] = _ym_index(out["year"], out["month"])

    empty = {
        "pathogen_within_5y": 0,
        "pathogen_within_10y": 0,
        "pathogen_detection_count_within_5y": 0,
        "pathogen_detection_count_within_10y": 0,
        "pathogen_ever_before": 0,
        "pathogen_ever_after": 0,
        "years_to_first_pathogen_after": pd.NA,
        "years_since_last_pathogen_before": pd.NA,
        "pathogen_same_month": 0,
        "pathogen_source_n_sequence": 0,
        "pathogen_source_n_outbreak": 0,
        "share_pathogen_exact_coords": pd.NA,
        "share_pathogen_subnational_centroid": pd.NA,
    }
    if viruses.empty:
        for col, val in empty.items():
            out[col] = val
        return out.drop(columns=["_row_id", "_p_ym"])

    vir = _prepare_virus_events(viruses)
    joined = out[
        ["_row_id", spatial_key, "vector_group", "pathogen_name", "_p_ym"]
    ].merge(
        vir[
            [
                spatial_key,
                "vector_group",
                "pathogen_name",
                "_v_ym",
                "record_id",
                "_insdc",
                "_outbreak",
                "_exact",
                "_centroid",
            ]
        ],
        on=[spatial_key, "vector_group", "pathogen_name"],
        how="left",
    )

    has_virus = joined["record_id"].notna()
    delta_m = joined["_v_ym"] - joined["_p_ym"]
    before = has_virus & (delta_m < 0)
    after = has_virus & (delta_m > 0)
    same = has_virus & (delta_m == 0)
    within_5 = after & (delta_m <= HORIZON_YEARS_PRIMARY * 12)
    within_10 = after & (delta_m <= HORIZON_YEARS_SECONDARY * 12)

    joined["_before"] = before.astype(int)
    joined["_after"] = after.astype(int)
    joined["_same"] = same.astype(int)
    joined["_w5"] = within_5.astype(int)
    joined["_w10"] = within_10.astype(int)
    joined["_delta_after"] = delta_m.where(after)
    joined["_delta_before"] = (-delta_m).where(before)
    joined["_insdc_w5"] = joined["_insdc"].where(within_5, 0)
    joined["_outbreak_w5"] = joined["_outbreak"].where(within_5, 0)
    joined["_exact_w5"] = joined["_exact"].where(within_5)
    joined["_centroid_w5"] = joined["_centroid"].where(within_5)

    agg = joined.groupby("_row_id", sort=False).agg(
        pathogen_ever_before=("_before", "max"),
        pathogen_ever_after=("_after", "max"),
        pathogen_same_month=("_same", "max"),
        pathogen_within_5y=("_w5", "max"),
        pathogen_within_10y=("_w10", "max"),
        pathogen_detection_count_within_5y=("_w5", "sum"),
        pathogen_detection_count_within_10y=("_w10", "sum"),
        _min_delta_after=("_delta_after", "min"),
        _min_delta_before=("_delta_before", "min"),
        pathogen_source_n_sequence=("_insdc_w5", "sum"),
        pathogen_source_n_outbreak=("_outbreak_w5", "sum"),
        share_pathogen_exact_coords=("_exact_w5", "mean"),
        share_pathogen_subnational_centroid=("_centroid_w5", "mean"),
    )

    for col in (
        "pathogen_ever_before",
        "pathogen_ever_after",
        "pathogen_same_month",
        "pathogen_within_5y",
        "pathogen_within_10y",
        "pathogen_detection_count_within_5y",
        "pathogen_detection_count_within_10y",
        "pathogen_source_n_sequence",
        "pathogen_source_n_outbreak",
    ):
        agg[col] = agg[col].fillna(0).astype(int)

    agg["years_to_first_pathogen_after"] = agg["_min_delta_after"] / 12.0
    agg["years_since_last_pathogen_before"] = agg["_min_delta_before"] / 12.0
    agg = agg.drop(columns=["_min_delta_after", "_min_delta_before"])

    if not include_geo_shares:
        agg["share_pathogen_exact_coords"] = pd.NA
        agg["share_pathogen_subnational_centroid"] = pd.NA

    out = out.merge(agg, left_on="_row_id", right_index=True, how="left")
    int_empty = {
        k
        for k, v in empty.items()
        if k
        in {
            "pathogen_ever_before",
            "pathogen_ever_after",
            "pathogen_same_month",
            "pathogen_within_5y",
            "pathogen_within_10y",
            "pathogen_detection_count_within_5y",
            "pathogen_detection_count_within_10y",
            "pathogen_source_n_sequence",
            "pathogen_source_n_outbreak",
        }
    }
    for col, val in empty.items():
        if col not in out.columns:
            out[col] = val
        elif col in int_empty:
            out[col] = out[col].fillna(val).astype(int)

    return out.drop(columns=["_row_id", "_p_ym"])


print("Helpers ready:", len(PRIORITY_SLUGS), "priority taxa,", len(pathogen_map), "pathogens")


## Build Tier 1 + Tier 2 panels (VectorNet-left × pathogen explosion + B2/A3 labels)

In [ ]:
vec_t1 = aggregate_vectors_tier1(vectors_t1)
vec_t1_long = explode_pathogens(vec_t1, pathogen_map)
panel_t1 = attach_b2_pathogen_labels(
    vec_t1_long,
    viruses_t1,
    spatial_key="h3_cell",
    include_geo_shares=True,
)

vec_t2 = aggregate_vectors_tier2(vectors_t2)
vec_t2_long = explode_pathogens(vec_t2, pathogen_map)
panel_t2 = attach_b2_pathogen_labels(
    vec_t2_long,
    viruses_t2,
    spatial_key="country_iso2",
    include_geo_shares=False,
)

panel = pd.concat([panel_t1, panel_t2], ignore_index=True, sort=False)
panel["vector_present"] = 1
panel["country_iso2"] = panel["country_iso2"].fillna("")

priority_cols = [f"vector_count_{slug}" for slug in PRIORITY_SLUGS]
column_order = [
    "spatial_tier",
    "h3_cell",
    "country_iso2",
    "year",
    "month",
    "vector_group",
    "pathogen_name",
    "vector_present",
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
    "species_richness",
    *priority_cols,
    "first_vector_year",
    "first_vector_month",
    "pathogen_within_5y",
    "pathogen_within_10y",
    "pathogen_detection_count_within_5y",
    "pathogen_detection_count_within_10y",
    "pathogen_ever_before",
    "pathogen_ever_after",
    "years_to_first_pathogen_after",
    "years_since_last_pathogen_before",
    "pathogen_same_month",
    "pathogen_source_n_sequence",
    "pathogen_source_n_outbreak",
    "share_pathogen_exact_coords",
    "share_pathogen_subnational_centroid",
]
panel = panel[column_order].sort_values(
    [
        "spatial_tier",
        "vector_group",
        "pathogen_name",
        "year",
        "month",
        "h3_cell",
        "country_iso2",
    ],
    kind="mergesort",
).reset_index(drop=True)

display(panel["spatial_tier"].value_counts())
display(
    panel.groupby(["spatial_tier", "vector_group", "pathogen_name"])
    .size()
    .rename("rows")
    .reset_index()
    .head(40)
)
display(
    panel.groupby("spatial_tier")
    .agg(
        rows=("vector_present", "size"),
        pathogen_within_5y=("pathogen_within_5y", "sum"),
        pathogen_within_10y=("pathogen_within_10y", "sum"),
        pathogen_same_month=("pathogen_same_month", "sum"),
        pathogen_ever_after=("pathogen_ever_after", "sum"),
        pathogen_ever_before=("pathogen_ever_before", "sum"),
        positive_rate_5y=("pathogen_within_5y", "mean"),
    )
)
panel.head()


## Manifest + write

In [ ]:
def _sha256(path: Path) -> str | None:
    if not path.exists():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


virus_manifest = default_virus_raw_dir(REPO_ROOT) / "manifest.json"
vector_manifest = default_vector_raw_dir(REPO_ROOT) / "manifest.json"

assert panel["vector_present"].eq(1).all()
assert set(panel["spatial_tier"].unique()) <= {"h3", "country"}
assert panel.loc[panel["spatial_tier"] == "h3", "h3_cell"].notna().all()
assert panel.loc[panel["spatial_tier"] == "country", "h3_cell"].isna().all()
assert panel.loc[panel["spatial_tier"] == "country", "country_iso2"].ne("").all()
assert panel["month"].between(1, 12).all()
assert panel["pathogen_name"].ne("").all()
assert not panel.duplicated(
    subset=["spatial_tier", "h3_cell", "country_iso2", "year", "month", "vector_group", "pathogen_name"]
).any()

# pathogen belongs to vector_group per map
check = panel[["pathogen_name", "vector_group"]].drop_duplicates().merge(
    pathogen_map, on=["pathogen_name", "vector_group"], how="left", indicator=True
)
assert (check["_merge"] == "both").all(), check.loc[check["_merge"] != "both"]

for col in (
    "pathogen_within_5y",
    "pathogen_within_10y",
    "pathogen_ever_before",
    "pathogen_ever_after",
    "pathogen_same_month",
):
    assert panel[col].isin([0, 1]).all()
assert "virus_within_5y" not in panel.columns
assert "virus_detected" not in panel.columns
assert (
    panel.loc[panel["pathogen_ever_after"] == 0, "years_to_first_pathogen_after"].isna().all()
)
assert (
    panel.loc[panel["pathogen_ever_before"] == 0, "years_since_last_pathogen_before"].isna().all()
)
assert (
    panel.loc[panel["pathogen_ever_after"] == 1, "years_to_first_pathogen_after"].notna().all()
)
assert (
    panel.loc[panel["pathogen_ever_before"] == 1, "years_since_last_pathogen_before"].notna().all()
)
assert (panel["pathogen_within_5y"] <= panel["pathogen_within_10y"]).all()
assert (panel["pathogen_within_10y"] <= panel["pathogen_ever_after"]).all()
for slug in PRIORITY_SLUGS:
    assert f"vector_count_{slug}" in panel.columns

# Non-spillover smoke: a mosquito unit with WNV history must not force dengue positive
# (structural: labels join on pathogen_name — verified by key uniqueness + join columns)

tier_counts = panel["spatial_tier"].value_counts().to_dict()
ym = panel.assign(_ym=panel["year"] * 100 + panel["month"])
ymin_row = ym.loc[ym["_ym"].idxmin()]
ymax_row = ym.loc[ym["_ym"].idxmax()]
date_min = f"{int(ymin_row['year'])}-{int(ymin_row['month']):02d}"
date_max = f"{int(ymax_row['year'])}-{int(ymax_row['month']):02d}"

pos_t1 = (panel["spatial_tier"] == "h3") & (panel["pathogen_within_5y"] == 1)
pathogen_geo_mix = {
    "tier1_rows_pathogen_within_5y": int(pos_t1.sum()),
    "tier1_rows_pathogen_within_10y": int(
        ((panel["spatial_tier"] == "h3") & (panel["pathogen_within_10y"] == 1)).sum()
    ),
    "tier1_rows_pathogen_same_month": int(
        ((panel["spatial_tier"] == "h3") & (panel["pathogen_same_month"] == 1)).sum()
    ),
    "mean_share_exact_coords_on_positive_tier1": float(
        panel.loc[pos_t1, "share_pathogen_exact_coords"].dropna().mean()
    )
    if pos_t1.any()
    else None,
    "mean_share_centroid_on_positive_tier1": float(
        panel.loc[pos_t1, "share_pathogen_subnational_centroid"].dropna().mean()
    )
    if pos_t1.any()
    else None,
}

positive_by_pathogen = (
    panel.groupby("pathogen_name")["pathogen_within_5y"]
    .agg(rows="size", positives="sum", positive_rate="mean")
    .reset_index()
)
positive_rates = {
    row.pathogen_name: {
        "rows": int(row.rows),
        "positives": int(row.positives),
        "positive_rate": float(row.positive_rate),
    }
    for row in positive_by_pathogen.itertuples(index=False)
}

manifest = {
    "H3_RESOLUTION": H3_RESOLUTION,
    "panel_rows": int(len(panel)),
    "rows_by_spatial_tier": {str(k): int(v) for k, v in tier_counts.items()},
    "eligibility": "from_first_vector",
    "primary_label": "pathogen_within_5y",
    "secondary_label": "pathogen_within_10y",
    "delay_missing": "null",
    "horizon_years_primary": HORIZON_YEARS_PRIMARY,
    "horizon_years_secondary": HORIZON_YEARS_SECONDARY,
    "n_pathogens": int(panel["pathogen_name"].nunique()),
    "pathogen_positive_units": int(panel["pathogen_within_5y"].sum()),
    "pathogen_positive_rate": float(panel["pathogen_within_5y"].mean()),
    "pathogen_within_10y_units": int(panel["pathogen_within_10y"].sum()),
    "pathogen_ever_after_units": int(panel["pathogen_ever_after"].sum()),
    "pathogen_ever_before_units": int(panel["pathogen_ever_before"].sum()),
    "pathogen_same_month_units": int(panel["pathogen_same_month"].sum()),
    "pathogen_positive_rates": positive_rates,
    "priority_feature_slugs": PRIORITY_SLUGS,
    "date_range": {"min": date_min, "max": date_max},
    "vector_groups": sorted(panel["vector_group"].unique().tolist()),
    "training_filter": (
        "Supervised panel includes only observed VectorNet months (B2 pathogen-long); "
        "virus-only units are excluded; no synthetic empty months after first vector (A3 clock only). "
        "Primary label pathogen_within_5y is forward from the row time in the same "
        "spatial key x pathogen_name (within vector_group). Zeros / null delays mean no reported "
        "detection of that pathogen in the merged feeds (right-censored by extract end), not proven absence."
    ),
    "temporal_grain": (
        "year x month (vector observation); pathogen labels relative to row timestamp"
    ),
    "tier1_key": ["h3_cell", "year", "month", "vector_group", "pathogen_name"],
    "tier2_key": ["country_iso2", "year", "month", "vector_group", "pathogen_name"],
    "pathogen_geo_precision_mix": pathogen_geo_mix,
    "inputs": {
        "vectors_normalized": "data/02_interim/vectors_normalized.parquet",
        "viruses_normalized": "data/02_interim/viruses_normalized.parquet",
        "pathogen_vector_group": "data/external/pathogen_vector_group.csv",
        "priority_vector_taxa": "data/external/priority_vector_taxa.csv",
    },
    "outputs": {
        "panel": "data/03_processed/modelling_panel.parquet",
        "manifest": "data/03_processed/modelling_panel_manifest.json",
    },
    "source_manifest_sha256": {
        "vector_manifest.json": _sha256(vector_manifest),
        "virus_manifest.json": _sha256(virus_manifest),
    },
}

panel.to_parquet(PANEL_PATH, index=False)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print(f"Wrote {PANEL_PATH} ({len(panel):,} rows)")
print(f"Wrote {MANIFEST_PATH}")
display(pd.Series({k: manifest[k] for k in [
    "panel_rows",
    "n_pathogens",
    "primary_label",
    "pathogen_positive_units",
    "pathogen_positive_rate",
    "pathogen_within_10y_units",
    "date_range",
]}))
